# Notebook 03: Logit Lens & Component Attribution (Level 2b)

## Overview
Applies **logit lens analysis** and **component attribution** to trace how Pythia models
build predictions for target tokens across frequency bands. The logit lens projects
intermediate residual stream states through the unembedding matrix to reveal when the
model "commits" to the correct prediction. Component attribution decomposes the logit
contribution into attention and MLP components at each layer.

## Key Questions
- At which layer does P(correct) become substantial? Does this differ by frequency band?
- When does the ranking of the correct token reach top-1?
- At which layer does the model's intermediate distribution converge to the final one (KL divergence)?
- Do low-frequency inputs converge later than high-frequency inputs?
- What fraction of the correct logit is attributable to attention vs MLP at each layer?
- How do these patterns scale across model sizes?

## Hypothesis Domain: R2b (Logit Lens & Attribution)
- **H-R2b.1**: Convergence layer differs by band (Kruskal-Wallis, per model)
- **H-R2b.2**: Low-frequency inputs converge later than high-frequency (Mann-Whitney, per model)
- **H-R2b.3**: Attention attribution fraction is higher for induction-circuit-reliant bands
- **H-R2b.4**: MLP attribution increases at deeper layers (signed attribution)
- **H-R2b.5**: Convergence layer scales with model depth (fractional convergence layer)

## Notebook Structure
1. Setup & Data Loading
2. Logit Lens: P(correct) Trajectory
3. Logit Lens: rank(correct) Trajectory
4. Logit Lens: KL from Final Layer
5. Convergence Layer Analysis
6. Component Attribution (Attention vs MLP)
7. Cross-Model Comparison
8. Summary

## Data Sources
- Pre-extracted activations from NPZ files: `logit_lens_prob_correct`, `logit_lens_rank_correct`,
  `logit_lens_kl_from_final`, `attn_out_predpos`, `mlp_out_predpos`, `target_ids`
- Shape per file: (N_examples, n_layers) for logit lens metrics
- Shape per file: (N_examples, n_layers, d_model) for component outputs

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    ACTIVATIONS_DIR,
    ANALYSIS_DIR,
    VIZ_DIR,
    RANDOM_SEED,
    CONVERGENCE_THRESHOLD,
    HF_MODEL_NAMES,
    get_domain_dirs,
)
from utils.data_loading import (
    load_extracted_activations,
    save_analysis,
    build_representational_df,
)
from utils.logit_lens import compute_convergence_layer, compute_component_attribution
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_logit_lens_heatmap,
    plot_metric_heatmap,
    plot_attribution_stacked,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("logit_lens", "base")
from functools import partial as _partial

save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")
print(f"Convergence threshold: {CONVERGENCE_THRESHOLD}")
print(f"Analysis dir: {ANALYSIS_DIR}")
print(f"Viz dir: {VIZ_DIR}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']
Convergence threshold: 0.9
Analysis dir: LSC_circuit_analysis/03_Phase_Representational/outputs/logit_lens/base/analysis
Viz dir: LSC_circuit_analysis/03_Phase_Representational/outputs/logit_lens/base/viz


In [2]:
# Load logit lens metrics and component outputs from pre-extracted NPZ files
all_prob_correct = {}  # model -> draw -> {band: (N, n_layers)}
all_rank_correct = {}  # model -> draw -> {band: (N, n_layers)}
all_kl_from_final = {}  # model -> draw -> {band: (N, n_layers)}
all_attn_out = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_mlp_out = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_target_ids = {}  # model -> draw -> {band: (N,)}

for model in MODELS:
    for store in [
        all_prob_correct,
        all_rank_correct,
        all_kl_from_final,
        all_attn_out,
        all_mlp_out,
        all_target_ids,
    ]:
        store[model] = {}
    for draw in DRAWS:
        for store in [
            all_prob_correct,
            all_rank_correct,
            all_kl_from_final,
            all_attn_out,
            all_mlp_out,
            all_target_ids,
        ]:
            store[model][draw] = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                all_prob_correct[model][draw][band] = data["logit_lens_prob_correct"]
                all_rank_correct[model][draw][band] = data["logit_lens_rank_correct"]
                all_kl_from_final[model][draw][band] = data["logit_lens_kl_from_final"]
                all_attn_out[model][draw][band] = data["attn_out_predpos"]
                all_mlp_out[model][draw][band] = data["mlp_out_predpos"]
                all_target_ids[model][draw][band] = data["target_ids"]
            except FileNotFoundError:
                pass

# Report what was loaded
for model in MODELS:
    n_loaded = sum(
        1
        for draw in DRAWS
        for band in BANDS
        if band in all_prob_correct[model].get(draw, {})
    )
    sample = next(
        (
            all_prob_correct[model][d][b]
            for d in DRAWS
            for b in BANDS
            if b in all_prob_correct[model].get(d, {})
        ),
        None,
    )
    if sample is not None:
        print(
            f"{model}: {n_loaded} configs loaded, "
            f"prob_correct shape = {sample.shape} (N, n_layers)"
        )
    else:
        print(f"{model}: no data loaded")

pythia-70m: 15 configs loaded, prob_correct shape = (225, 6) (N, n_layers)
pythia-160m: 15 configs loaded, prob_correct shape = (225, 12) (N, n_layers)
pythia-410m: 15 configs loaded, prob_correct shape = (225, 24) (N, n_layers)
pythia-1b: 15 configs loaded, prob_correct shape = (225, 16) (N, n_layers)
pythia-1.4b: 15 configs loaded, prob_correct shape = (225, 24) (N, n_layers)


## 2. Logit Lens: P(correct) Trajectory

How does the probability assigned to the correct target token evolve through layers?
We plot per-band trajectories for each model and produce a layers x bands heatmap.

In [3]:
prob_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            prob = all_prob_correct[model].get(draw, {}).get(band)
            if prob is None:
                continue
            # prob shape: (N, n_layers)
            mean_prob = prob.mean(axis=0)  # (n_layers,)
            std_prob = prob.std(axis=0)
            median_prob = np.median(prob, axis=0)

            for layer in range(n_layers):
                prob_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_prob_correct": float(mean_prob[layer]),
                        "std_prob_correct": float(std_prob[layer]),
                        "median_prob_correct": float(median_prob[layer]),
                    }
                )

df_prob = pd.DataFrame(prob_records)
save_analysis(df_prob, "03_prob_correct_trajectory.csv")
print(f"P(correct) trajectory records: {len(df_prob)}")
print(f"\nFinal-layer mean P(correct) by model x band (draw_1):")
final_prob = (
    df_prob[(df_prob["draw"] == "draw_1")]
    .groupby(["model", "band"])
    .apply(lambda g: g.loc[g["layer"].idxmax(), "mean_prob_correct"])
    .unstack("band")
)
print(final_prob.round(3))

P(correct) trajectory records: 1230

Final-layer mean P(correct) by model x band (draw_1):
band         control   high    low  medium  very_high
model                                                
pythia-1.4b    0.540  0.533  0.539   0.534      0.544
pythia-160m    0.517  0.505  0.407   0.457      0.562
pythia-1b      0.610  0.632  0.649   0.712      0.582
pythia-410m    0.509  0.561  0.600   0.588      0.534
pythia-70m     0.226  0.209  0.101   0.171      0.262


In [4]:
# Plot P(correct) trajectory per model, colored by band
for model in MODELS:
    model_data = df_prob[(df_prob["model"] == model) & (df_prob["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 6))
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)
        ax.plot(
            bd["layer"],
            bd["mean_prob_correct"],
            color=color,
            label=label,
            marker="o",
            markersize=3,
        )
        ax.fill_between(
            bd["layer"],
            bd["mean_prob_correct"] - bd["std_prob_correct"],
            bd["mean_prob_correct"] + bd["std_prob_correct"],
            color=color,
            alpha=0.12,
        )

    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean P(correct)")
    ax.set_title(f"Logit Lens: P(correct) Trajectory \u2014 {model}")
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)
    save_figure(fig, f"viz_03_01_prob_correct_trajectory_{model}.png")

In [5]:
# Heatmap: bands x layers for each model (P(correct) averaged across draws)
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    model_data = df_prob[df_prob["model"] == model]
    if len(model_data) == 0:
        continue

    # Average across draws
    avg = (
        model_data.groupby(["layer", "band"])["mean_prob_correct"].mean().reset_index()
    )
    pivot = avg.pivot(index="band", columns="layer", values="mean_prob_correct")
    pivot = pivot.reindex(index=[b for b in BANDS if b in pivot.index])
    pivot = pivot.reindex(columns=list(range(n_layers)))

    bands_present = [b for b in BANDS if b in pivot.index]
    layers = list(range(n_layers))
    values = pivot.values

    fig = plot_logit_lens_heatmap(
        values,
        bands_present,
        layers,
        title=f"P(correct) by Layer: {model}",
        cmap="YlOrRd",
        fmt=".2f",
        xlabel="Layer",
        ylabel="Band",
    )
    save_figure(fig, f"viz_03_02_prob_correct_heatmap_{model}.png")

## 3. Logit Lens: rank(correct) Trajectory

How does the ranking of the correct token (0 = top-ranked) evolve through layers?
Lower rank means the correct token is higher in the model's prediction.

In [6]:
rank_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            rank = all_rank_correct[model].get(draw, {}).get(band)
            if rank is None:
                continue
            mean_rank = rank.mean(axis=0).astype(float)
            median_rank = np.median(rank, axis=0).astype(float)
            # Fraction at rank 0 (top-1)
            frac_top1 = (rank == 0).mean(axis=0).astype(float)
            frac_top5 = (rank < 5).mean(axis=0).astype(float)

            for layer in range(n_layers):
                rank_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_rank": float(mean_rank[layer]),
                        "median_rank": float(median_rank[layer]),
                        "frac_top1": float(frac_top1[layer]),
                        "frac_top5": float(frac_top5[layer]),
                    }
                )

df_rank = pd.DataFrame(rank_records)
save_analysis(df_rank, "03_rank_correct_trajectory.csv")
print(f"Rank trajectory records: {len(df_rank)}")
print(f"\nFinal-layer mean rank by model x band (draw_1):")
final_rank = (
    df_rank[(df_rank["draw"] == "draw_1")]
    .groupby(["model", "band"])
    .apply(lambda g: g.loc[g["layer"].idxmax(), "mean_rank"])
    .unstack("band")
)
print(final_rank.round(1))

Rank trajectory records: 1230

Final-layer mean rank by model x band (draw_1):
band         control   high     low  medium  very_high
model                                                 
pythia-1.4b      1.2    0.0     0.1     0.2        0.2
pythia-160m      0.0    0.1     0.2     0.1        0.0
pythia-1b        0.0    0.0     0.1    10.7        0.0
pythia-410m      0.0    0.0     0.0     0.0        0.0
pythia-70m     186.0  687.0  2140.6  1177.4       71.4


In [7]:
# Plot mean rank trajectory per model
for model in MODELS:
    model_data = df_rank[(df_rank["model"] == model) & (df_rank["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Left: mean rank (log scale)
    ax = axes[0]
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        color = BAND_COLORS.get(band, "gray")
        ax.plot(
            bd["layer"],
            bd["mean_rank"],
            color=color,
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean Rank of Correct Token")
    ax.set_title(f"rank(correct) Trajectory \u2014 {model}")
    ax.set_yscale("symlog", linthresh=1)
    ax.legend(fontsize=7)

    # Right: fraction top-1
    ax = axes[1]
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        color = BAND_COLORS.get(band, "gray")
        ax.plot(
            bd["layer"],
            bd["frac_top1"],
            color=color,
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Fraction Top-1")
    ax.set_title(f"Fraction Correct is Top-1 \u2014 {model}")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7)

    fig.tight_layout()
    save_figure(fig, f"viz_03_03_rank_trajectory_{model}.png")

In [8]:
# Heatmap: bands x layers for median rank
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    model_data = df_rank[df_rank["model"] == model]
    if len(model_data) == 0:
        continue

    avg = model_data.groupby(["layer", "band"])["median_rank"].mean().reset_index()
    pivot = avg.pivot(index="band", columns="layer", values="median_rank")
    pivot = pivot.reindex(index=[b for b in BANDS if b in pivot.index])
    pivot = pivot.reindex(columns=list(range(n_layers)))

    bands_present = [b for b in BANDS if b in pivot.index]
    layers = list(range(n_layers))
    values = pivot.values

    fig = plot_logit_lens_heatmap(
        values,
        bands_present,
        layers,
        title=f"Median Rank(correct) by Layer: {model}",
        cmap="YlOrRd_r",
        fmt=".0f",
        xlabel="Layer",
        ylabel="Band",
        annot=False,
    )
    save_figure(fig, f"viz_03_04_rank_heatmap_{model}.png")

## 4. Logit Lens: KL from Final Layer

KL(final-layer distribution || layer-l distribution) measures how much the
intermediate prediction diverges from the final one. When this drops to near zero,
the model has effectively "committed" to its final answer.

In [9]:
kl_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            kl = all_kl_from_final[model].get(draw, {}).get(band)
            if kl is None:
                continue
            mean_kl = kl.mean(axis=0)
            median_kl = np.median(kl, axis=0)
            std_kl = kl.std(axis=0)

            for layer in range(n_layers):
                kl_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_kl": float(mean_kl[layer]),
                        "median_kl": float(median_kl[layer]),
                        "std_kl": float(std_kl[layer]),
                    }
                )

df_kl = pd.DataFrame(kl_records)
save_analysis(df_kl, "03_kl_from_final_trajectory.csv")
print(f"KL trajectory records: {len(df_kl)}")
print(f"\nMean KL at first layer by model x band (draw_1):")
first_kl = df_kl[(df_kl["draw"] == "draw_1") & (df_kl["layer"] == 0)].pivot(
    index="model", columns="band", values="mean_kl"
)
print(first_kl.round(2))

KL trajectory records: 1230

Mean KL at first layer by model x band (draw_1):
band         control   high    low  medium  very_high
model                                                
pythia-1.4b     7.70   8.08   9.13    8.61       7.80
pythia-160m    17.06  16.69  16.47   16.63      17.32
pythia-1b       8.23   8.52  10.14   10.13       8.12
pythia-410m     9.80  10.17  11.05   10.40       9.90
pythia-70m     16.00  15.72  14.72   15.54      16.36


In [10]:
# Plot KL trajectory per model
for model in MODELS:
    model_data = df_kl[(df_kl["model"] == model) & (df_kl["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 6))
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        color = BAND_COLORS.get(band, "gray")
        ax.plot(
            bd["layer"],
            bd["mean_kl"],
            color=color,
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
        ax.fill_between(
            bd["layer"],
            np.maximum(0, bd["mean_kl"] - bd["std_kl"]),
            bd["mean_kl"] + bd["std_kl"],
            color=color,
            alpha=0.12,
        )

    ax.set_xlabel("Layer")
    ax.set_ylabel("KL(final || layer) (nats)")
    ax.set_title(f"KL Divergence from Final Layer \u2014 {model}")
    ax.legend(fontsize=8)
    ax.set_yscale("symlog", linthresh=0.1)
    save_figure(fig, f"viz_03_05_kl_trajectory_{model}.png")

In [11]:
# KL heatmap: bands x layers (log scale)
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    model_data = df_kl[df_kl["model"] == model]
    if len(model_data) == 0:
        continue

    avg = model_data.groupby(["layer", "band"])["mean_kl"].mean().reset_index()
    pivot = avg.pivot(index="band", columns="layer", values="mean_kl")
    pivot = pivot.reindex(index=[b for b in BANDS if b in pivot.index])
    pivot = pivot.reindex(columns=list(range(n_layers)))

    bands_present = [b for b in BANDS if b in pivot.index]
    layers = list(range(n_layers))
    values = np.log10(pivot.values + 1e-10)

    fig = plot_logit_lens_heatmap(
        values,
        bands_present,
        layers,
        title=f"log10 KL(final || layer): {model}",
        cmap="YlOrRd_r",
        fmt=".1f",
        xlabel="Layer",
        ylabel="Band",
    )
    save_figure(fig, f"viz_03_06_kl_heatmap_{model}.png")

## 5. Convergence Layer Analysis

The convergence layer is the first layer where P(correct) exceeds
`CONVERGENCE_THRESHOLD` (default 0.9) times the final-layer P(correct).
This measures when the model has built up enough of its final prediction.

Key question: **Do low-frequency inputs converge later?**

In [12]:
convergence_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            prob = all_prob_correct[model].get(draw, {}).get(band)
            if prob is None:
                continue

            # Compute per-example convergence layer
            conv_layers = compute_convergence_layer(
                prob, threshold=CONVERGENCE_THRESHOLD
            )  # (N,)

            # Filter out examples that never converged
            converged_mask = conv_layers < n_layers
            n_converged = converged_mask.sum()
            n_total = len(conv_layers)

            if n_converged > 0:
                conv_vals = conv_layers[converged_mask]
                mean_conv = float(conv_vals.mean())
                median_conv = float(np.median(conv_vals))
                std_conv = float(conv_vals.std())
            else:
                mean_conv = float(n_layers)
                median_conv = float(n_layers)
                std_conv = 0.0

            convergence_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "band": band,
                    "mean_convergence_layer": mean_conv,
                    "median_convergence_layer": median_conv,
                    "std_convergence_layer": std_conv,
                    "frac_converged": float(n_converged / n_total),
                    "n_converged": int(n_converged),
                    "n_total": int(n_total),
                    "n_layers": n_layers,
                    "frac_convergence_layer": mean_conv / n_layers,
                }
            )

df_conv = build_representational_df(convergence_records)
save_analysis(df_conv, "03_convergence_layers.csv")
print(f"Convergence records: {len(df_conv)}")
print(f"\nMean convergence layer by model x band:")
print(
    df_conv.groupby(["model", "band"])["mean_convergence_layer"]
    .mean()
    .unstack("band")
    .round(2)
)
print(f"\nFraction converged by model x band:")
print(
    df_conv.groupby(["model", "band"])["frac_converged"].mean().unstack("band").round(3)
)

Convergence records: 75

Mean convergence layer by model x band:
band           low  medium   high  very_high  control
model                                                
pythia-70m    4.72    4.63   4.51       4.34     4.36
pythia-160m   9.55    9.22   8.95       8.67     8.69
pythia-410m  21.01   20.12  18.86      17.56    17.84
pythia-1b    12.49   11.92  11.41      11.00    11.16
pythia-1.4b  21.57   19.37  16.79      14.61    15.15

Fraction converged by model x band:
band         low  medium  high  very_high  control
model                                             
pythia-70m   1.0     1.0   1.0        1.0      1.0
pythia-160m  1.0     1.0   1.0        1.0      1.0
pythia-410m  1.0     1.0   1.0        1.0      1.0
pythia-1b    1.0     1.0   1.0        1.0      1.0
pythia-1.4b  1.0     1.0   1.0        1.0      1.0


In [13]:
# Heatmap: mean convergence layer (model x band)
conv_pivot = (
    df_conv.groupby(["model", "band"])["mean_convergence_layer"].mean().reset_index()
)
conv_wide = conv_pivot.pivot(
    index="model", columns="band", values="mean_convergence_layer"
)
conv_wide = conv_wide.reindex(
    index=MODELS, columns=[b for b in BANDS if b in conv_wide.columns]
)

fig = plot_metric_heatmap(
    conv_wide,
    title=f"Mean Convergence Layer (threshold={CONVERGENCE_THRESHOLD})",
    fmt=".1f",
    cmap="YlOrRd",
)
save_figure(fig, "viz_03_07_convergence_heatmap.png")

In [14]:
# Boxplot: convergence layer by band for each model
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]

    # Collect per-example convergence layers for draw_1
    per_example_records = []
    for band in BANDS:
        prob = all_prob_correct[model].get("draw_1", {}).get(band)
        if prob is None:
            continue
        conv = compute_convergence_layer(prob, threshold=CONVERGENCE_THRESHOLD)
        for c in conv:
            per_example_records.append(
                {
                    "band": band,
                    "convergence_layer": int(c),
                }
            )

    if not per_example_records:
        continue

    df_ex = pd.DataFrame(per_example_records)

    fig, ax = plt.subplots(figsize=(10, 6))
    palette = {b: BAND_COLORS[b] for b in BANDS if b in BAND_COLORS}
    order = [b for b in BANDS if b in df_ex["band"].unique()]

    sns.boxplot(
        data=df_ex,
        x="band",
        y="convergence_layer",
        palette=palette,
        order=order,
        ax=ax,
        fliersize=2,
    )
    sns.stripplot(
        data=df_ex,
        x="band",
        y="convergence_layer",
        palette=palette,
        order=order,
        ax=ax,
        alpha=0.15,
        size=2,
        jitter=True,
    )

    ax.set_xlabel("Frequency Band")
    ax.set_ylabel("Convergence Layer")
    ax.set_title(f"Per-Example Convergence Layer \u2014 {model}")
    ax.axhline(
        y=n_layers, color="red", linestyle="--", alpha=0.3, label="Never converged"
    )
    ax.legend(fontsize=8)
    save_figure(fig, f"viz_03_08_convergence_boxplot_{model}.png")

<TMPDIR>/ipykernel_241414/1955224400.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
<TMPDIR>/ipykernel_241414/1955224400.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


<TMPDIR>/ipykernel_241414/1955224400.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
<TMPDIR>/ipykernel_241414/1955224400.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


<TMPDIR>/ipykernel_241414/1955224400.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


<TMPDIR>/ipykernel_241414/1955224400.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


<TMPDIR>/ipykernel_241414/1955224400.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
<TMPDIR>/ipykernel_241414/1955224400.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


<TMPDIR>/ipykernel_241414/1955224400.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
<TMPDIR>/ipykernel_241414/1955224400.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


In [15]:
# Statistical tests: band differences in convergence layer
from scipy import stats as sp_stats

stat_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        # Collect per-example convergence per band
        band_conv = {}
        for band in BANDS:
            prob = all_prob_correct[model].get(draw, {}).get(band)
            if prob is None:
                continue
            conv = compute_convergence_layer(prob, threshold=CONVERGENCE_THRESHOLD)
            band_conv[band] = conv.astype(float)

        if len(band_conv) < 2:
            continue

        # Kruskal-Wallis across all bands
        groups = [band_conv[b] for b in BANDS if b in band_conv]
        if all(len(g) > 0 for g in groups):
            h_stat, h_pval = sp_stats.kruskal(*groups)
            stat_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "test": "kruskal_wallis",
                    "statistic": float(h_stat),
                    "p_value": float(h_pval),
                    "description": "Convergence layer differs by band",
                }
            )

        # Mann-Whitney: low vs high frequency
        low_bands = ["low"]
        high_bands = ["very_high"]
        low_vals = np.concatenate([band_conv[b] for b in low_bands if b in band_conv])
        high_vals = np.concatenate([band_conv[b] for b in high_bands if b in band_conv])

        if len(low_vals) > 0 and len(high_vals) > 0:
            u_stat, u_pval = sp_stats.mannwhitneyu(
                low_vals, high_vals, alternative="greater"
            )
            stat_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "test": "mann_whitney_low_vs_veryhigh",
                    "statistic": float(u_stat),
                    "p_value": float(u_pval),
                    "description": "Low-freq converges later than very_high",
                }
            )

        # Spearman correlation: frequency rank vs convergence layer
        # Skip bands with None rank (e.g. 'control')
        all_conv = []
        all_freq_rank = []
        for band in BANDS:
            if band in band_conv and FREQUENCY_RANK.get(band) is not None:
                all_conv.extend(band_conv[band].tolist())
                all_freq_rank.extend([FREQUENCY_RANK[band]] * len(band_conv[band]))

        if len(all_conv) > 10:
            rho, rho_p = sp_stats.spearmanr(all_freq_rank, all_conv)
            stat_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "test": "spearman_freqrank_vs_conv",
                    "statistic": float(rho),
                    "p_value": float(rho_p),
                    "description": "Frequency rank correlates with convergence layer",
                }
            )

df_stats = pd.DataFrame(stat_records)
save_analysis(df_stats, "03_convergence_stats.csv")
print("Convergence statistical tests:")
for test_name in df_stats["test"].unique():
    subset = df_stats[df_stats["test"] == test_name]
    print(f"\n  {test_name}: {subset.iloc[0]['description']}")
    for _, row in subset.iterrows():
        sig = "*" if row["p_value"] < 0.05 else ""
        print(
            f"    {row['model']}/{row['draw']}: stat={row['statistic']:.3f}, "
            f"p={row['p_value']:.4f} {sig}"
        )

Convergence statistical tests:

  kruskal_wallis: Convergence layer differs by band
    pythia-70m/draw_1: stat=46.448, p=0.0000 *
    pythia-70m/draw_2: stat=53.686, p=0.0000 *
    pythia-70m/draw_3: stat=86.605, p=0.0000 *
    pythia-160m/draw_1: stat=47.929, p=0.0000 *
    pythia-160m/draw_2: stat=51.542, p=0.0000 *
    pythia-160m/draw_3: stat=66.444, p=0.0000 *
    pythia-410m/draw_1: stat=276.538, p=0.0000 *
    pythia-410m/draw_2: stat=245.182, p=0.0000 *
    pythia-410m/draw_3: stat=279.987, p=0.0000 *
    pythia-1b/draw_1: stat=106.673, p=0.0000 *
    pythia-1b/draw_2: stat=106.161, p=0.0000 *
    pythia-1b/draw_3: stat=124.240, p=0.0000 *
    pythia-1.4b/draw_1: stat=336.065, p=0.0000 *
    pythia-1.4b/draw_2: stat=390.087, p=0.0000 *
    pythia-1.4b/draw_3: stat=409.086, p=0.0000 *

  mann_whitney_low_vs_veryhigh: Low-freq converges later than very_high
    pythia-70m/draw_1: stat=32376.000, p=0.0000 *
    pythia-70m/draw_2: stat=31892.500, p=0.0000 *
    pythia-70m/draw_3: 

## 6. Component Attribution (Attention vs MLP)

Decompose the correct-token logit into attention and MLP contributions at each layer:

- `attn_contrib[l] = attn_out[l] . W_U[:, target_id]`
- `mlp_contrib[l]  = mlp_out[l]  . W_U[:, target_id]`

**Important**: We do NOT apply `ln_final` to individual components, as LayerNorm is
a nonlinear operation that should only be applied to the full residual stream.

This section requires the model's unembedding matrix `W_U`. We attempt to load the
model; if unavailable (no GPU), we fall back to a norm-based proxy.

In [16]:
# Attempt to load W_U from models for proper attribution
# If no GPU / model loading fails, use norm-based proxy

USE_WU_ATTRIBUTION = False
model_W_U = {}  # model -> W_U numpy array (d_model, d_vocab)

try:
    from utils.extraction import load_model
    import gc

    for model_name in MODELS:
        try:
            print(f"Loading {model_name} for W_U extraction...")
            model_obj = load_model(model_name, device="cpu", verbose=False)
            W_U = model_obj.W_U.detach().cpu().numpy()  # (d_model, d_vocab)
            model_W_U[model_name] = W_U
            print(f"  W_U shape: {W_U.shape}")
            del model_obj
            gc.collect()
        except Exception as e:
            print(f"  Failed to load {model_name}: {e}")

    if len(model_W_U) == len(MODELS):
        USE_WU_ATTRIBUTION = True
        print(f"\nAll models loaded. Using W_U-based attribution.")
    elif len(model_W_U) > 0:
        USE_WU_ATTRIBUTION = True
        print(
            f"\nPartially loaded ({len(model_W_U)}/{len(MODELS)}). "
            f"Using W_U for available models, norm proxy for the rest."
        )
    else:
        print(f"\nNo models loaded. Falling back to norm-based proxy.")

except ImportError:
    print("transformer_lens not available. Using norm-based proxy for attribution.")
except Exception as e:
    print(f"Model loading failed: {e}. Using norm-based proxy.")

print(f"\nUSE_WU_ATTRIBUTION = {USE_WU_ATTRIBUTION}")

Loading pythia-70m for W_U extraction...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-70m into HookedTransformer
  W_U shape: (512, 50304)
Loading pythia-160m for W_U extraction...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Loaded pretrained model EleutherAI/pythia-1b into HookedTransformer
  W_U shape: (2048, 50304)


Loading pythia-1.4b for W_U extraction...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-1.4b into HookedTransformer
  W_U shape: (2048, 50304)



All models loaded. Using W_U-based attribution.

USE_WU_ATTRIBUTION = True


In [17]:
# Compute component attribution
attrib_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    W_U = model_W_U.get(model)  # May be None

    for draw in DRAWS:
        for band in BANDS:
            attn = all_attn_out[model].get(draw, {}).get(band)
            mlp = all_mlp_out[model].get(draw, {}).get(band)
            tids = all_target_ids[model].get(draw, {}).get(band)

            if attn is None or mlp is None or tids is None:
                continue

            if USE_WU_ATTRIBUTION and W_U is not None:
                # Proper attribution via dot product with W_U
                result = compute_component_attribution(attn, mlp, W_U, tids)
                attn_logit = result["attn_logit"]  # (N, n_layers)
                mlp_logit = result["mlp_logit"]  # (N, n_layers)
                attn_frac = result["attn_frac"]  # (N, n_layers)
                mlp_frac = result["mlp_frac"]  # (N, n_layers)
                method = "W_U"
            else:
                # Norm-based proxy: ||attn_out|| / (||attn_out|| + ||mlp_out||)
                attn_norms = np.linalg.norm(attn, axis=-1)  # (N, n_layers)
                mlp_norms = np.linalg.norm(mlp, axis=-1)  # (N, n_layers)
                total_norms = attn_norms + mlp_norms + 1e-10
                attn_frac = attn_norms / total_norms
                mlp_frac = mlp_norms / total_norms
                attn_logit = attn_norms  # Proxy: use norms in place of logit contrib
                mlp_logit = mlp_norms
                method = "norm_proxy"

            # Average across examples
            for layer in range(n_layers):
                attrib_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "attn_logit_mean": float(attn_logit[:, layer].mean()),
                        "mlp_logit_mean": float(mlp_logit[:, layer].mean()),
                        "attn_frac": float(attn_frac[:, layer].mean()),
                        "mlp_frac": float(mlp_frac[:, layer].mean()),
                        "attn_logit_std": float(attn_logit[:, layer].std()),
                        "mlp_logit_std": float(mlp_logit[:, layer].std()),
                        "method": method,
                    }
                )

df_attrib = pd.DataFrame(attrib_records)
save_analysis(df_attrib, "03_component_attribution.csv")
print(f"Attribution records: {len(df_attrib)}")
print(f"Method used: {df_attrib['method'].unique()}")
print(f"\nMean attribution fractions (draw_1, final layer):")
final_attrib = (
    df_attrib[df_attrib["draw"] == "draw_1"]
    .groupby(["model", "band"])
    .apply(
        lambda g: pd.Series(
            {
                "attn_frac": g.loc[g["layer"].idxmax(), "attn_frac"],
                "mlp_frac": g.loc[g["layer"].idxmax(), "mlp_frac"],
            }
        )
    )
    .reset_index()
)
print(final_attrib.pivot(index="model", columns="band", values="attn_frac").round(3))

Attribution records: 1230
Method used: <ArrowStringArray>
['W_U']
Length: 1, dtype: str

Mean attribution fractions (draw_1, final layer):
band         control   high    low  medium  very_high
model                                                
pythia-1.4b    0.228  0.191  0.154   0.158      0.257
pythia-160m    0.190  0.190  0.196   0.193      0.187
pythia-1b      0.436  0.419  0.375   0.376      0.460
pythia-410m    0.177  0.162  0.145   0.151      0.183
pythia-70m     0.335  0.342  0.351   0.351      0.331


In [18]:
# Stacked area plot: attention vs MLP attribution across layers per band
for model in MODELS:
    model_data = df_attrib[
        (df_attrib["model"] == model) & (df_attrib["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    fig = plot_attribution_stacked(
        model_data,
        model=model,
        title="Attention vs MLP Attribution",
    )
    save_figure(fig, f"viz_03_09_attribution_stacked_{model}.png")

In [19]:
# Signed logit contribution trajectories (attn and MLP raw logit contributions)
for model in MODELS:
    model_data = df_attrib[
        (df_attrib["model"] == model) & (df_attrib["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    n_bands = len([b for b in BANDS if b in model_data["band"].unique()])
    fig, axes = plt.subplots(1, n_bands, figsize=(4 * n_bands, 5), sharey=True)
    if n_bands == 1:
        axes = [axes]

    for ax, band in zip(axes, [b for b in BANDS if b in model_data["band"].unique()]):
        bd = model_data[model_data["band"] == band].sort_values("layer")
        layers = bd["layer"].values

        ax.plot(
            layers,
            bd["attn_logit_mean"],
            color="#1f77b4",
            label="Attention",
            marker="o",
            markersize=3,
        )
        ax.fill_between(
            layers,
            bd["attn_logit_mean"] - bd["attn_logit_std"],
            bd["attn_logit_mean"] + bd["attn_logit_std"],
            color="#1f77b4",
            alpha=0.12,
        )

        ax.plot(
            layers,
            bd["mlp_logit_mean"],
            color="#ff7f0e",
            label="MLP",
            marker="s",
            markersize=3,
        )
        ax.fill_between(
            layers,
            bd["mlp_logit_mean"] - bd["mlp_logit_std"],
            bd["mlp_logit_mean"] + bd["mlp_logit_std"],
            color="#ff7f0e",
            alpha=0.12,
        )

        ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
        ax.set_xlabel("Layer")
        ax.set_title(BAND_NAMES.get(band, band))

    axes[0].set_ylabel("Logit Contribution to Correct Token")
    axes[-1].legend(fontsize=8)
    fig.suptitle(f"Signed Component Attribution \u2014 {model}", y=1.02)
    fig.tight_layout()
    save_figure(fig, f"viz_03_10_attribution_signed_{model}.png")

In [20]:
# Heatmap: attention fraction (model x band, averaged over layers and draws)
attrib_summary = (
    df_attrib.groupby(["model", "band"])
    .agg(
        {
            "attn_frac": "mean",
            "mlp_frac": "mean",
        }
    )
    .reset_index()
)

for metric, title, cmap in [
    ("attn_frac", "Mean Attention Attribution Fraction", "Blues"),
    ("mlp_frac", "Mean MLP Attribution Fraction", "Oranges"),
]:
    pivot = attrib_summary.pivot(index="model", columns="band", values=metric)
    pivot = pivot.reindex(
        index=MODELS, columns=[b for b in BANDS if b in pivot.columns]
    )
    fig = plot_metric_heatmap(pivot, title, fmt=".3f", cmap=cmap)
    suffix = metric.replace("_frac", "")
    save_figure(fig, f"viz_03_11_attribution_{suffix}_heatmap.png")

## 7. Cross-Model Comparison

Compare convergence layers and attribution profiles across model sizes.
Do larger models converge at the same fractional depth? Does the attention/MLP
balance shift with model scale?

In [21]:
# Build master logit lens DataFrame
master_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            record = {
                "model": model,
                "draw": draw,
                "band": band,
                "model_capacity": MODEL_CAPACITY[model],
                "n_layers": n_layers,
            }

            # Convergence
            conv_row = df_conv[
                (df_conv["model"] == model)
                & (df_conv["draw"] == draw)
                & (df_conv["band"] == band)
            ]
            if len(conv_row) > 0:
                record["convergence_layer"] = conv_row.iloc[0]["mean_convergence_layer"]
                record["frac_convergence"] = conv_row.iloc[0]["frac_convergence_layer"]
                record["frac_converged"] = conv_row.iloc[0]["frac_converged"]

            # Final-layer P(correct)
            prob_final = df_prob[
                (df_prob["model"] == model)
                & (df_prob["draw"] == draw)
                & (df_prob["band"] == band)
            ]
            if len(prob_final) > 0:
                record["final_prob_correct"] = prob_final.loc[
                    prob_final["layer"].idxmax(), "mean_prob_correct"
                ]

            # Final-layer rank
            rank_final = df_rank[
                (df_rank["model"] == model)
                & (df_rank["draw"] == draw)
                & (df_rank["band"] == band)
            ]
            if len(rank_final) > 0:
                record["final_mean_rank"] = rank_final.loc[
                    rank_final["layer"].idxmax(), "mean_rank"
                ]
                record["final_frac_top1"] = rank_final.loc[
                    rank_final["layer"].idxmax(), "frac_top1"
                ]

            # Mean attribution (across layers)
            attrib_data = df_attrib[
                (df_attrib["model"] == model)
                & (df_attrib["draw"] == draw)
                & (df_attrib["band"] == band)
            ]
            if len(attrib_data) > 0:
                record["mean_attn_frac"] = attrib_data["attn_frac"].mean()
                record["mean_mlp_frac"] = attrib_data["mlp_frac"].mean()

            master_records.append(record)

df_master = build_representational_df(master_records)
save_analysis(df_master, "03_master_logit_lens.csv")
print(f"Master records: {len(df_master)}")
print(f"\nKey metrics by model (averaged across bands and draws):")
summary_cols = [
    "final_prob_correct",
    "final_mean_rank",
    "convergence_layer",
    "frac_convergence",
    "mean_attn_frac",
]
existing_cols = [c for c in summary_cols if c in df_master.columns]
print(df_master.groupby("model")[existing_cols].mean().round(3))

Master records: 75

Key metrics by model (averaged across bands and draws):
             final_prob_correct  final_mean_rank  convergence_layer  \
model                                                                 
pythia-70m                0.201          902.584              4.512   
pythia-160m               0.488            0.100              9.016   
pythia-410m               0.562            0.024             19.081   
pythia-1b                 0.648            0.746             11.595   
pythia-1.4b               0.553            0.202             17.500   

             frac_convergence  mean_attn_frac  
model                                          
pythia-70m              0.752           0.496  
pythia-160m             0.751           0.400  
pythia-410m             0.795           0.531  
pythia-1b               0.725           0.598  
pythia-1.4b             0.729           0.559  


In [22]:
# Fractional convergence layer across models, per band
if "frac_convergence" in df_master.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    for band in BANDS:
        band_data = df_master[df_master["band"] == band]
        if len(band_data) == 0:
            continue
        means = band_data.groupby("model")["frac_convergence"].mean()
        caps = band_data.groupby("model")["model_capacity"].first()
        color = BAND_COLORS.get(band, "gray")
        ax.plot(
            caps,
            means,
            color=color,
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=6,
        )

    ax.set_xlabel("Model Capacity (M params)")
    ax.set_ylabel("Fractional Convergence Layer (layer / n_layers)")
    ax.set_title("Convergence Depth Across Model Sizes")
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    save_figure(fig, "viz_03_12_convergence_scaling.png")

In [23]:
# Attribution balance across model sizes
if "mean_attn_frac" in df_master.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: attention fraction by band across model sizes
    ax = axes[0]
    for band in BANDS:
        band_data = df_master[df_master["band"] == band]
        if len(band_data) == 0:
            continue
        means = band_data.groupby("model")["mean_attn_frac"].mean()
        caps = band_data.groupby("model")["model_capacity"].first()
        color = BAND_COLORS.get(band, "gray")
        ax.plot(
            caps,
            means,
            color=color,
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=6,
        )
    ax.set_xlabel("Model Capacity (M params)")
    ax.set_ylabel("Mean Attention Attribution Fraction")
    ax.set_title("Attention Attribution vs Model Size")
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)

    # Right: final P(correct) by band across model sizes
    ax = axes[1]
    if "final_prob_correct" in df_master.columns:
        for band in BANDS:
            band_data = df_master[df_master["band"] == band]
            if len(band_data) == 0:
                continue
            means = band_data.groupby("model")["final_prob_correct"].mean()
            caps = band_data.groupby("model")["model_capacity"].first()
            color = BAND_COLORS.get(band, "gray")
            ax.plot(
                caps,
                means,
                color=color,
                label=BAND_NAMES.get(band, band),
                marker="o",
                markersize=6,
            )
        ax.set_xlabel("Model Capacity (M params)")
        ax.set_ylabel("Final P(correct)")
        ax.set_title("Final Prediction Quality vs Model Size")
        ax.set_xscale("log")
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1)

    fig.tight_layout()
    save_figure(fig, "viz_03_13_cross_model_scaling.png")

In [24]:
# Overlay all models: P(correct) trajectory using fractional depth (layer / n_layers)
fig, ax = plt.subplots(figsize=(12, 6))

for model in MODELS:
    model_data = df_prob[
        (df_prob["model"] == model)
        & (df_prob["draw"] == "draw_1")
        & (df_prob["band"] == "low")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    frac_depth = model_data["layer"].values / n_layers
    color = MODEL_COLORS.get(model, "gray")
    ax.plot(
        frac_depth,
        model_data["mean_prob_correct"],
        color=color,
        label=model,
        marker="o",
        markersize=3,
    )

ax.set_xlabel("Fractional Depth (layer / n_layers)")
ax.set_ylabel("Mean P(correct)")
ax.set_title("P(correct) vs Fractional Depth \u2014 Low Frequency Band")
ax.legend(fontsize=8)
ax.set_ylim(bottom=0)
save_figure(fig, "viz_03_14_prob_fractional_depth.png")

In [25]:
# Draw stability for key metrics
stability_records = []

for model in MODELS:
    for band in BANDS:
        subset = df_master[(df_master["model"] == model) & (df_master["band"] == band)]
        if len(subset) < 2:
            continue

        for metric in [
            "convergence_layer",
            "frac_convergence",
            "final_prob_correct",
            "mean_attn_frac",
        ]:
            if metric not in subset.columns:
                continue
            vals = subset[metric].dropna()
            if len(vals) < 2:
                continue
            stability_records.append(
                {
                    "model": model,
                    "band": band,
                    "metric": metric,
                    "mean": float(vals.mean()),
                    "std": float(vals.std()),
                    "cv": float(vals.std() / abs(vals.mean()))
                    if abs(vals.mean()) > 1e-10
                    else np.nan,
                }
            )

df_stability = pd.DataFrame(stability_records)
save_analysis(df_stability, "03_draw_stability.csv")
print("Draw stability (CV by model, averaged across bands):")
if len(df_stability) > 0:
    print(
        df_stability.groupby(["model", "metric"])["cv"]
        .mean()
        .unstack("metric")
        .round(3)
    )

Draw stability (CV by model, averaged across bands):
metric       convergence_layer  final_prob_correct  frac_convergence  \
model                                                                  
pythia-1.4b              0.012               0.029             0.012   
pythia-160m              0.013               0.012             0.013   
pythia-1b                0.004               0.031             0.004   
pythia-410m              0.007               0.027             0.007   
pythia-70m               0.010               0.083             0.010   

metric       mean_attn_frac  
model                        
pythia-1.4b           0.005  
pythia-160m           0.011  
pythia-1b             0.006  
pythia-410m           0.007  
pythia-70m            0.018  


## 8. Summary

Consolidate key findings from logit lens and attribution analysis.

In [26]:
print("=" * 70)
print("NOTEBOOK 03: LOGIT LENS & COMPONENT ATTRIBUTION: SUMMARY")
print("=" * 70)

# 1. Convergence layer differences
print("\n--- Convergence Layer Analysis ---")
print(f"Convergence threshold: {CONVERGENCE_THRESHOLD}")
kw_tests = (
    df_stats[df_stats["test"] == "kruskal_wallis"]
    if len(df_stats) > 0
    else pd.DataFrame()
)
if len(kw_tests) > 0:
    n_sig = (kw_tests["p_value"] < 0.05).sum()
    print(
        f"Kruskal-Wallis (band effect on convergence): "
        f"{n_sig}/{len(kw_tests)} significant at p<0.05"
    )

mw_tests = (
    df_stats[df_stats["test"] == "mann_whitney_low_vs_veryhigh"]
    if len(df_stats) > 0
    else pd.DataFrame()
)
if len(mw_tests) > 0:
    n_sig = (mw_tests["p_value"] < 0.05).sum()
    print(
        f"Mann-Whitney (low > very_high convergence layer): "
        f"{n_sig}/{len(mw_tests)} significant at p<0.05"
    )

# 2. Attribution balance
print("\n--- Component Attribution ---")
print(f"Method: {df_attrib['method'].unique() if len(df_attrib) > 0 else 'N/A'}")
if "mean_attn_frac" in df_master.columns:
    for model in MODELS:
        model_data = df_master[df_master["model"] == model]
        if "mean_attn_frac" in model_data.columns and len(model_data) > 0:
            attn_mean = model_data["mean_attn_frac"].mean()
            mlp_mean = model_data["mean_mlp_frac"].mean()
            print(f"  {model}: attn={attn_mean:.3f}, mlp={mlp_mean:.3f}")

# 3. Final prediction quality
print("\n--- Final-Layer Prediction Quality ---")
if "final_prob_correct" in df_master.columns:
    summary = df_master.groupby("model")["final_prob_correct"].mean()
    for model, val in summary.items():
        print(f"  {model}: mean P(correct) = {val:.3f}")

# 4. Cross-model scaling
print("\n--- Cross-Model Scaling ---")
if "frac_convergence" in df_master.columns:
    frac_summary = df_master.groupby("model")["frac_convergence"].mean()
    for model, val in frac_summary.items():
        print(f"  {model}: mean fractional convergence = {val:.3f}")

# Output listing
print("\n" + "=" * 70)
print("NOTEBOOK 03 COMPLETE")
print("=" * 70)
print(f"\nOutput CSVs in: {ANALYSIS_DIR}")
print(f"Figures in: {VIZ_DIR}")
for f in sorted(ANALYSIS_DIR.glob("03_*")):
    print(f"  {f.name}")
for f in sorted(VIZ_DIR.glob("viz_03_*")):
    print(f"  {f.name}")

NOTEBOOK 03: LOGIT LENS & COMPONENT ATTRIBUTION: SUMMARY

--- Convergence Layer Analysis ---
Convergence threshold: 0.9
Kruskal-Wallis (band effect on convergence): 15/15 significant at p<0.05
Mann-Whitney (low > very_high convergence layer): 15/15 significant at p<0.05

--- Component Attribution ---
Method: <ArrowStringArray>
['W_U']
Length: 1, dtype: str
  pythia-70m: attn=0.496, mlp=0.504
  pythia-160m: attn=0.400, mlp=0.600
  pythia-410m: attn=0.531, mlp=0.469
  pythia-1b: attn=0.598, mlp=0.402
  pythia-1.4b: attn=0.559, mlp=0.441

--- Final-Layer Prediction Quality ---
  pythia-70m: mean P(correct) = 0.201
  pythia-160m: mean P(correct) = 0.488
  pythia-410m: mean P(correct) = 0.562
  pythia-1b: mean P(correct) = 0.648
  pythia-1.4b: mean P(correct) = 0.553

--- Cross-Model Scaling ---
  pythia-70m: mean fractional convergence = 0.752
  pythia-160m: mean fractional convergence = 0.751
  pythia-410m: mean fractional convergence = 0.795
  pythia-1b: mean fractional convergence = 0.7